# 01 Item-level replication adapters (seeds 43-46) and a PeftModel re-evaluation

The original runs saved only the seed-42 item-level adapter. This notebook retrains seeds 43-46
with the same data split and hyperparameters as the main pipeline and saves the adapters (they are
the `replication/` adapters on the Hugging Face Hub). It then loads the base model and all five
adapters with `PeftModel.from_pretrained` and evaluates them in the original option order on the
five benchmarks, comparing every prediction with the released `finetuned_seed<N>_*.csv`.

The `plain_*` files written here are read by notebooks 02 and 03 (Section 4.9 of the paper).

Run the cells in order: [0-1] installs the pinned package versions; restart the runtime
(Runtime > Restart session) and continue from [0-2]. Results are written to Google Drive, and
cells that find their output already saved skip it, so an interrupted run can be resumed by
running [0-2] again and then the interrupted cell.

`DRIVE_DIR` is the Drive folder used by `notebooks/tombench_slm_qlora_complete_pipeline.ipynb`;
`tier4_v2_fixed/` inside it holds that notebook's outputs (split and manifest files, per-item
CSVs, adapters). The other folders are created by the revision notebooks; see
`notebooks/revision/README.md` for where their files are in `results/`.

## [0] Setup

In [ ]:
# -- [0-1] Install packages (versions pinned to requirements.lock.txt) --
import sys, os, subprocess, time
import torch
if not torch.cuda.is_available():
    print('\nNo GPU detected. Runtime -> Change runtime type -> select a GPU.'); raise SystemExit
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'GPU: {gpu_name} ({vram_gb:.1f}GB VRAM)')

print('\nInstalling packages (2-3 min)...')
t0 = time.time()
# Install the exact combination recorded with pip freeze in requirements.lock.txt.
# Installing unsloth alone pulls the latest unsloth, which can pull a newer transformers
# in which TrainingArguments no longer accepts warmup_ratio, among other API changes.
# Keep the versions below.
get_ipython().system('pip install -q unsloth==2026.8.18 unsloth_zoo==2026.8.12')
get_ipython().system('pip install -q transformers==5.5.0 trl==0.24.0 accelerate==1.14.0 bitsandbytes==0.50.1 peft==0.20.0')
get_ipython().system('pip install -q "datasets==3.6.0" tqdm')
print(f'   {time.time()-t0:.0f}s')
import transformers, trl, unsloth
print(f'unsloth={unsloth.__version__} transformers={transformers.__version__} trl={trl.__version__}')
assert transformers.__version__ == '5.5.0', f'transformers version not pinned: {transformers.__version__}'


In [ ]:
# -- [0-2] Mount Google Drive, set paths --
from google.colab import drive
drive.mount('/content/drive')

import json as _json, glob, re, random, string, hashlib
import pandas as pd
import numpy as np

DRIVE_DIR   = '/content/drive/MyDrive/ToMBench_Results'
RESULTS_DIR = f'{DRIVE_DIR}/tier4_v2_fixed'
OUT_DIR_P2  = f'{DRIVE_DIR}/cyclic_shift_check'
OUT_DIR_T   = f'{DRIVE_DIR}/optrand_check'
OUT_DIR_S   = f'{DRIVE_DIR}/storydisjoint_multiseed'
for d in [OUT_DIR_P2, OUT_DIR_T, OUT_DIR_S]:
    os.makedirs(d, exist_ok=True)

BENCHMARKS = ['ToMBench', 'ToMi', 'OpenToM', 'SocialIQa', 'HiToM']
SEEDS = [42, 43, 44, 45, 46]
MODEL_NAME = 'unsloth/Qwen2.5-3B-Instruct-bnb-4bit'
MAX_SEQ_LENGTH = 2048
NUM_EPOCHS = 4
EARLY_STOPPING_PATIENCE = 2
print('Paths ready')


In [ ]:
# -- [0-3] Helper functions (same as the main pipeline notebook) --
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
from unsloth import is_bfloat16_supported
from peft import PeftModel

SYSTEM_PROMPT = (
    'Below is a multiple-choice question with a story and several answer options. '
    'Based on the content of the story and the given question, please infer the most likely answer '
    'and output the answer index in the format [[X]] where X is one of A, B, C, D, E.'
)

LETTERS = list(string.ascii_uppercase)
ANS_RE = re.compile(r'\[\[\s*([A-E])\s*\]\]', re.IGNORECASE)
FB_RE = re.compile(r'\b([A-E])\b')

def stable_hash(s) -> int:
    return int(hashlib.sha256(str(s).encode('utf-8')).hexdigest(), 16) & 0xffffffff

def prompt_hash_of(user_prompt) -> str:
    return hashlib.sha256(str(user_prompt).encode('utf-8')).hexdigest()[:16]

def canonical_story_id(story) -> str:
    return hashlib.sha256(' '.join(str(story).split()).encode('utf-8')).hexdigest()[:16]

def build_user_prompt(story, question, options):
    lines = [f'{l}. {t}' for l, t in options]
    return f'[Story]\n{story}\n\n[Question]\n{question}\n\n[Candidate Answers]\n' + '\n'.join(lines)

def parse_options(user_prompt):
    m = re.search(r'\[Candidate Answers\]\n(.+)', user_prompt, re.S)
    block = m.group(1) if m else user_prompt
    return [(l, t.strip()) for l, t in re.findall(r'^([A-E])\.\s*(.+?)$', block, re.M)]

def extract_letter(text):
    m = ANS_RE.search(text)
    if m: return m.group(1).upper()
    m = FB_RE.search(text)
    if m: return m.group(1).upper()
    return 'A'

def to_text(tokenizer, user_prompt):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user_prompt}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def to_sft_text(tokenizer, user_prompt, gold):
    msgs = [{'role': 'system', 'content': SYSTEM_PROMPT}, {'role': 'user', 'content': user_prompt},
            {'role': 'assistant', 'content': f'[[{gold}]]'}]
    return tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)

def load_fresh_model():
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
    return model, tokenizer

def get_batch_sizes():
    _gpu = torch.cuda.get_device_name(0)
    _vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if 'A100' in _gpu or _vram >= 35: return 8, 1, 16
    elif 'L4' in _gpu or _vram >= 22: return 4, 2, 8
    else: return 2, 4, 4

def make_training_args(seed, output_dir):
    train_bs, grad_accum, eval_bs = get_batch_sizes()
    return TrainingArguments(
        per_device_train_batch_size=train_bs, per_device_eval_batch_size=eval_bs,
        gradient_accumulation_steps=grad_accum, warmup_ratio=0.03, num_train_epochs=NUM_EPOCHS,
        learning_rate=2e-4, fp16=not is_bfloat16_supported(), bf16=is_bfloat16_supported(),
        logging_steps=20, eval_strategy='epoch', save_strategy='epoch', load_best_model_at_end=True,
        metric_for_best_model='eval_loss', greater_is_better=False, save_total_limit=2,
        optim='adamw_8bit', weight_decay=0.01, lr_scheduler_type='cosine', seed=seed,
        output_dir=output_dir, report_to='none',
    )

def train_qlora(train_records, val_records, seed, output_dir):
    """Train a QLoRA adapter on lists of records with user_prompt and answer and
    return (model, tokenizer). The model stays in memory; the caller deletes it
    and calls torch.cuda.empty_cache() after evaluation."""
    model, tokenizer = load_fresh_model()
    model = FastLanguageModel.get_peft_model(
        model, r=16,
        target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
        lora_alpha=16, lora_dropout=0.0, bias='none',
        use_gradient_checkpointing='unsloth', random_state=seed,
    )
    train_texts = [to_sft_text(tokenizer, r['user_prompt'], r['answer']) for r in train_records]
    val_texts   = [to_sft_text(tokenizer, r['user_prompt'], r['answer']) for r in val_records]
    train_ds = Dataset.from_dict({'text': train_texts})
    val_ds   = Dataset.from_dict({'text': val_texts})

    trainer = SFTTrainer(
        model=model, tokenizer=tokenizer, train_dataset=train_ds, eval_dataset=val_ds,
        dataset_text_field='text', max_seq_length=MAX_SEQ_LENGTH, dataset_num_proc=2, packing=False,
        args=make_training_args(seed, output_dir),
        callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    )
    t0 = time.time()
    trainer.train()
    print(f'Training done ({(time.time()-t0)/60:.1f} min)')
    del trainer
    return model, tokenizer

@torch.no_grad()
def evaluate(eval_records, model, tokenizer, desc='Eval', max_new_tokens=12):
    from tqdm.auto import tqdm
    FastLanguageModel.for_inference(model)
    results = []
    for r in tqdm(eval_records, desc=desc):
        prompt = to_text(tokenizer, r['user_prompt'])
        inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
        gen = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        pred = extract_letter(gen)
        results.append({
            'item_id': r['item_id'], 'prompt_hash': r.get('prompt_hash'),
            'source': r.get('source'), 'task': r.get('task'), 'category': r.get('category'),
            'ability': r.get('ability'), 'answer': r['answer'], 'pred': pred, 'raw': gen,
            'correct': int(pred == r['answer']), 'n_options': r.get('n_options'),
            'marker_found': '[[' in gen,
        })
    return pd.DataFrame(results)

def load_manifest(bm):
    path = f'{RESULTS_DIR}/split_test_full_v2.json' if bm == 'ToMBench' else f'{RESULTS_DIR}/manifest_{bm}.json'
    with open(path, encoding='utf-8') as f:
        items = _json.load(f)
    for it in items:
        if 'options' not in it:
            it['options'] = parse_options(it['user_prompt'])
    return items

with open(f'{RESULTS_DIR}/split_train_full_v2.json', encoding='utf-8') as f:
    ITEM_TRAIN = _json.load(f)
with open(f'{RESULTS_DIR}/split_val_full_v2.json', encoding='utf-8') as f:
    ITEM_VAL = _json.load(f)
with open(f'{RESULTS_DIR}/split_test_full_v2.json', encoding='utf-8') as f:
    ITEM_TEST = _json.load(f)

print('Helpers ready. Item-level split:', len(ITEM_TRAIN), len(ITEM_VAL), len(ITEM_TEST))


## [A] Replication adapters and original-order re-evaluation

In [ ]:
# -- [A-2] Retrain seeds 43-46 and save the adapters (skipped if already saved) --
adapter_paths_p2 = {42: f'{RESULTS_DIR}/adapter_seed42'}

for s in [43, 44, 45, 46]:
    adapter_path = f'{RESULTS_DIR}/adapter_seed{s}'
    if os.path.isdir(adapter_path) and os.path.exists(f'{adapter_path}/adapter_model.safetensors'):
        print(f'[skip] seed {s}: adapter already saved')
        adapter_paths_p2[s] = adapter_path
        continue
    print('='*60); print(f'Re-training seed {s}'); print('='*60)
    model, tokenizer = train_qlora(ITEM_TRAIN, ITEM_VAL, s, f'outputs_seed{s}_readapt')
    model.save_pretrained(adapter_path)
    tokenizer.save_pretrained(adapter_path)
    print(f'Saved -> {adapter_path}')
    adapter_paths_p2[s] = adapter_path
    del model, tokenizer
    torch.cuda.empty_cache()

print('\nAdapters ready:', adapter_paths_p2)


In [ ]:
# -- [A-3] Load the base model and the five seed adapters --
p2_base_model, p2_tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
FastLanguageModel.for_inference(p2_base_model)

p2_tuned_models = {}
for s in SEEDS:
    _b, _ = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LENGTH, load_in_4bit=True)
    p2_tuned_models[s] = PeftModel.from_pretrained(_b, adapter_paths_p2[s])
    FastLanguageModel.for_inference(p2_tuned_models[s])
print('Base model and five seed adapters loaded')


### [A-3-GATE] Original-order re-evaluation

Evaluates the base model once per benchmark and each seed adapter, saves the predictions as
`plain_base_<benchmark>.csv` and `plain_seed<N>_<benchmark>.csv`, and compares them item by item
with the released CSVs. Seed 42 is the adapter that produced the released files, so mismatches for
seed 42 come from the evaluation, while seeds 43-46 also include retraining.

In [ ]:
# -- [A-3-GATE] Evaluate seeds 42-46 in the original option order and compare with the released CSVs --
# Results already saved are loaded instead of re-evaluated, so the cell can be rerun.
mismatch_rows = []
delta_rows = []

for bm in BENCHMARKS:
    items = load_manifest(bm)

    base_path = f'{OUT_DIR_P2}/plain_base_{bm}.csv'
    if os.path.exists(base_path):
        print(f'[skip] base {bm}: already saved')
        df_base = pd.read_csv(base_path)
    else:
        df_base = evaluate(items, p2_base_model, p2_tokenizer, desc=f'gate-base-{bm}')
        df_base.to_csv(base_path, index=False)
    new_base_acc = df_base['correct'].mean() * 100

    # released base predictions (for the original change)
    orig_base_path = f'{RESULTS_DIR}/baseline_{bm}.csv'
    if os.path.exists(orig_base_path):
        orig_base_acc = pd.read_csv(orig_base_path)['correct'].mean() * 100
    else:
        print(f'[warning] released base CSV missing: {orig_base_path}')
        orig_base_acc = None

    for s in SEEDS:  # [42, 43, 44, 45, 46]
        tuned_path = f'{OUT_DIR_P2}/plain_seed{s}_{bm}.csv'
        if os.path.exists(tuned_path):
            print(f'[skip] seed{s} {bm}: already saved')
            df_new = pd.read_csv(tuned_path)
        else:
            df_new = evaluate(items, p2_tuned_models[s], p2_tokenizer, desc=f'gate-seed{s}-{bm}')
            df_new.to_csv(tuned_path, index=False)
        new_tuned_acc = df_new['correct'].mean() * 100

        old_path = f'{RESULTS_DIR}/finetuned_seed{s}_{bm}.csv'
        if os.path.exists(old_path):
            df_old = pd.read_csv(old_path)
            merged = df_new.merge(df_old, on='item_id', suffixes=('_new', '_old'))
            n_mismatch = int((merged['pred_new'] != merged['pred_old']).sum())
            n_total = len(merged)
            orig_tuned_acc = df_old['correct'].mean() * 100
        else:
            print(f'[warning] released CSV missing: {old_path}')
            n_mismatch, n_total, orig_tuned_acc = None, len(df_new), None

        new_delta = new_tuned_acc - new_base_acc
        orig_delta = (orig_tuned_acc - orig_base_acc) if (orig_tuned_acc is not None and orig_base_acc is not None) else None
        delta_diff = (new_delta - orig_delta) if orig_delta is not None else None

        mismatch_rows.append({'benchmark': bm, 'seed': s, 'mismatch': n_mismatch, 'n_items': n_total})
        delta_rows.append({
            'benchmark': bm, 'seed': s,
            'new_base_acc': new_base_acc, 'new_tuned_acc': new_tuned_acc, 'new_delta_pp': new_delta,
            'orig_base_acc': orig_base_acc, 'orig_tuned_acc': orig_tuned_acc, 'orig_delta_pp': orig_delta,
            'delta_diff_pp': delta_diff,
        })
        orig_delta_str = 'N/A' if orig_delta is None else f'{orig_delta:+.2f}'
        diff_str = 'N/A' if delta_diff is None else f'{delta_diff:+.2f}'
        print(f'[{bm}] seed{s}: mismatches {n_mismatch}/{n_total} | new Δ={new_delta:+.2f}pp, orig Δ={orig_delta_str}pp, diff={diff_str}pp')

df_mismatch = pd.DataFrame(mismatch_rows)
df_delta = pd.DataFrame(delta_rows)
df_mismatch.to_csv(f'{OUT_DIR_P2}/gating_mismatch_summary.csv', index=False)
df_delta.to_csv(f'{OUT_DIR_P2}/gating_aggregate_delta.csv', index=False)

print('\n=== Prediction mismatches against the released CSVs (benchmark x seed) ===')
print(df_mismatch.pivot(index='benchmark', columns='seed', values='mismatch'))

print('\n=== Change from base: re-evaluation vs released results (pp) ===')
print(df_delta[['benchmark', 'seed', 'new_delta_pp', 'orig_delta_pp', 'delta_diff_pp']].to_string(index=False))

print('\nA non-zero seed-42 mismatch means that this evaluation differs from the one that produced the released files.')
print('delta_diff_pp near 0 means the aggregate change is preserved.')

### [A-3-VERIFY] Checks on the comparison

Reads only the saved CSVs. The first cell checks that every join covers the expected number of
items; the second shows a few mismatching items.

In [ ]:
# -- [A-3-VERIFY-1] Row counts of the joins (a join error would change n_items) --
df_mismatch_check = pd.read_csv(f'{OUT_DIR_P2}/gating_mismatch_summary.csv')
expected_n = {'ToMBench': 842, 'ToMi': 1000, 'OpenToM': 1000, 'SocialIQa': 1000, 'HiToM': 1000}
bad = df_mismatch_check[df_mismatch_check.apply(lambda r: r['n_items'] != expected_n[r['benchmark']], axis=1)]
print(bad if len(bad) else 'All joins complete: n_items equals the expected item count for every file')

In [ ]:
# -- [A-3-VERIFY-2] Generated text of a few mismatching items --
CHECK_BENCHMARK = 'HiToM'
CHECK_SEED = 42

df_old_sample = pd.read_csv(f'{RESULTS_DIR}/finetuned_seed{CHECK_SEED}_{CHECK_BENCHMARK}.csv')
df_new_sample = pd.read_csv(f'{OUT_DIR_P2}/plain_seed{CHECK_SEED}_{CHECK_BENCHMARK}.csv')
m = df_new_sample.merge(df_old_sample, on='item_id', suffixes=('_new', '_old'))
diff = m[m['pred_new'] != m['pred_old']]
print(f'{len(diff)} mismatching items (seed{CHECK_SEED}, {CHECK_BENCHMARK})')
print(diff[['item_id', 'pred_new', 'pred_old', 'raw_new', 'raw_old']].head(5).to_string())